<a href="https://colab.research.google.com/github/DCC773/Predicci-n-de-la-recesi-n-en-el-Per-/blob/main/Eleccion_binaria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


TITULO:
Predicción de Recesión en Perú (2000–2024)

1)INTRODUCCION:

En este informe se presenta una aplicación de modelos de elección binaria para predecir la ocurrencia de recesiones en el Perú durante el periodo 2000–2024.

El análisis se centra en construir un modelo de clasificación binaria, en donde la variable dependiente es la presencia o no de una recesión (RECESION = 1 si hubo recesión, 0 en caso contrario). Para lograr esto, se emplean como variables explicativas indicadores macroeconómicos clave:

- **Inflación (% anual)**
- **Crecimiento del Producto Bruto Interno (PBI, % anual)**
- **Tasa de desempleo (% de la población económicamente activa)**

El modelo utilizado para esta tarea es una **red neuronal artificial** que utiliza una **función de activación sigmoide**, común en problemas de clasificación binaria. Esta elección permite capturar relaciones no lineales entre las variables y la probabilidad de que ocurra una recesión.

 3. CARGA Y PREPARACION DE DATOS

In [ ]:
import numpy as np
import pandas as pd

# Datos reales del Perú 2000–2024
años = np.arange(2000, 2025)
pbi = [2.7, 0.6, 5.5, 4.2, 5, 6.3, 7.5, 8.5, 9.1, 1.1,
       8.3, 6.3, 6.1, 5.9, 2.4, 3.3, 4.0, 2.5, 4.0, 2.2,
       -11.0, 13.4, 2.8, -0.4, 3.3]
inflacion = [3.8, 2.0, 0.2, 2.3, 3.7, 1.6, 2.0, 1.8, 5.8, 2.9,
             1.5, 3.37, 3.61, 2.77, 3.41, 3.40, 3.56, 2.99, 1.51, 2.25,
             2.00, 4.27, 8.33, 6.46, 2.01]
desempleo = [5.0, 5.1, 4.8, 4.2, 4.7, 4.9, 4.2, 4.1, 4.0, 4.0,
             3.6, 3.5, 3.2, 3.6, 3.2, 3.3, 3.7, 3.7, 3.5, 3.4,
             7.2, 5.1, 3.9, 4.9, 4.8]
recesion = [0]*20 + [1, 0, 0, 1, 0]  # Recesiones en 2020 y 2023

# Crear DataFrame
df = pd.DataFrame({
    'AÑO': años,
    'INFLACION': inflacion,
    'PBI': pbi,
    'DESEMPLEO': desempleo,
    'RECESION': recesion
})
df.head()
df

,AÑO,INFLACION,PBI,DESEMPLEO,RECESION
0,2000,3.80,2.7,5.0,0
1,2001,2.00,0.6,5.1,0
2,2002,0.20,5.5,4.8,0
3,2003,2.30,4.2,4.2,0
4,2004,3.70,5.0,4.7,0
5,2005,1.60,6.3,4.9,0
6,2006,2.00,7.5,4.2,0
7,2007,1.80,8.5,4.1,0
8,2008,5.80,9.1,4.0,0
9,2009,2.90,1.1,4.0,0


5. CONSTRUCCION DEL MODELO



El modelo consta de una red neuronal secuencial con las siguientes características:

- Una **capa oculta** con 8 neuronas y función de activación **ReLU**
- Una **capa de salida** con 1 neurona y función de activación **sigmoide**, que produce una probabilidad entre 0 y 1
- **Función de pérdida**: `binary_crossentropy`, adecuada para clasificación binaria
- **Optimizador**: `adam`, eficiente y ampliamente usado


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Variables independientes y dependiente
X = df[['INFLACION', 'PBI', 'DESEMPLEO']]
y = df['RECESION']

# Escalamiento
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# División en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y)




# Modelo
model = Sequential()
model.add(Dense(8, activation='relu', input_shape=(3,)))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=150, verbose=0, validation_data=(X_test, y_test))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


6. EVALUACION DEL MODELO

Se evalúa el modelo utilizando el conjunto de prueba. Se genera un reporte de clasificación que incluye:

- **Precision**: Qué tan precisas son las predicciones positivas
- **Recall**: Qué proporción de casos positivos reales se detectaron
- **F1-score**: Media armónica entre precisión y recall

Además, se muestra el resultado en formato tabular para una mejor interpretación.

In [ ]:
from sklearn.metrics import classification_report

# Predicción
y_pred = (model.predict(X_test) > 0.5).astype(int)

# Reporte
report = classification_report(y_test, y_pred, output_dict=True)
tabla = pd.DataFrame(report).transpose()
tabla.round(2)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,precision,recall,f1-score,support
0,0.88,1.00,0.93,7.00
1,0.00,0.00,0.00,1.00
accuracy,0.88,0.88,0.88,0.88
macro avg,0.44,0.50,0.47,8.00
weighted avg,0.77,0.88,0.82,8.00


El modelo tiene un buen desempeño clasificando los años sin recesión, pero no logra identificar correctamente los años con recesión (2020 y 2023). Esto puede deberse al desequilibrio en la base de datos: hay muchos más años sin recesión (23 años) que con recesión (solo 2 años), lo cual afecta el entrenamiento de la red neuronal.